# PHASE 2: CREDIT FEATURE ENGINEERING
I will filter to resolved loans, which exlcudes 'current' or in progress loans so that the model may only learn from  completed outcomes. 
We will look at 'FULLY PAY' vs 'DEFAULT'

In [2]:
import pandas as pd
import numpy as np

In [3]:
#Loading the streamlined dataset
df = pd.read_csv('../data/processed/cleaned_loans_streamlined.csv')
print(f'Loaded {df.shape[0]:,} rows | {df.shape[1]} columns\n')

Loaded 2,260,668 rows | 24 columns



In [4]:
print(df["loan_status"].value_counts(dropna=False))

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64


In [6]:
#We must define what is good status and what is bad status
good_statuses = [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid"
]

bad_statuses = [
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off"
]

#creating model 
df_model = df[df["loan_status"].isin(good_statuses + bad_statuses)].copy()
df_model["is_default"] = df_model["loan_status"].isin(bad_statuses).astype(int)


print(f"Resolved loans shape: {df_model.shape[0]:,} rows")
print("\nDefault Rate (%):")
print((df_model["is_default"].value_counts(normalize=True) * 100).round(2))

Resolved loans shape: 1,348,099 rows

Default Rate (%):
is_default
0    80.02
1    19.98
Name: proportion, dtype: float64


In [ ]:
# clip income to 1 to prevent division by zero
safe_income = df_model["annual_inc"].clip(lower= 1)

df_model["loan_to_income"] = df_model["loan_amnt"]/ safe_income
df_model["installment_to_income"] = df_model["installment"] /(safe_income / 12)


,loan_amnt,annual_inc,loan_to_income,installment_to_income,is_default
0,3600.0,55000.0,0.065455,0.026843,0
1,24700.0,65000.0,0.380000,0.151436,0
2,20000.0,63000.0,0.317460,0.082411,0
4,10400.0,104433.0,0.099585,0.033312,0
5,11950.0,34000.0,0.351471,0.143005,0


In [8]:
df_model[['loan_amnt', 'annual_inc', 'loan_to_income', 'installment_to_income', 'is_default']].head(25)

,loan_amnt,annual_inc,loan_to_income,installment_to_income,is_default
0,3600.0,55000.0,0.065455,0.026843,0
1,24700.0,65000.0,0.380000,0.151436,0
2,20000.0,63000.0,0.317460,0.082411,0
4,10400.0,104433.0,0.099585,0.033312,0
5,11950.0,34000.0,0.351471,0.143005,0
6,20000.0,180000.0,0.111111,0.042505,0
7,20000.0,85000.0,0.235294,0.089119,0
8,10000.0,85000.0,0.117647,0.043264,0
9,8000.0,42000.0,0.190476,0.075354,0
12,1400.0,64000.0,0.021875,0.008831,0
